# Enterprise System Health Analytics

**Goal:** turn infrastructure telemetry into operational insights, anomaly flags, health scores, and an executive risk summary.

> Dataset is synthetic and intended for portfolio/demo use.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_metrics
from src.validation import clean_metrics, validate_ranges
from src.anomaly_detection import add_anomaly_flags
from src.health_scoring import calculate_health_score

RAW = ROOT / "data" / "raw" / "system_metrics.csv"
PROCESSED = ROOT / "data" / "processed" / "cleaned_metrics.csv"
df = load_metrics(RAW)
df.head()

## 1. Data quality assessment

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.isna().sum().to_frame("missing_values"))
display(df.describe(include="all").T)

## 2. Cleaning and validation

In [ ]:
clean = clean_metrics(df)
print("Range validation:", validate_ranges(clean))
clean.to_csv(PROCESSED, index=False)
clean.head()

## 3. Infrastructure utilization

In [ ]:
metrics = ["cpu_pct","memory_pct","disk_pct"]
clean[metrics].mean().sort_values(ascending=False).plot(kind="bar", figsize=(8,4), title="Average Resource Utilization")
plt.ylabel("Percent")
plt.tight_layout()
plt.show()

## 4. Latency and application errors

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.scatter(clean["network_latency_ms"], clean["error_rate_pct"], alpha=0.25, s=8)
ax.set_xlabel("Network latency (ms)")
ax.set_ylabel("Error rate (%)")
ax.set_title("Latency vs Application Error Rate")
plt.tight_layout()
plt.show()

## 5. Anomaly detection

In [ ]:
analyzed = add_anomaly_flags(clean)
print(f"Anomalous observations: {analyzed['anomaly_flag'].sum():,} ({analyzed['anomaly_flag'].mean()*100:.2f}%)")
display(analyzed.loc[analyzed["anomaly_flag"], ["system_id","network_latency_ms","error_rate_pct","latency_z","error_z"]].head(10))

## 6. Health scoring and risk classification

In [ ]:
analyzed = calculate_health_score(analyzed)
risk_summary = analyzed["risk_level"].value_counts().reindex(["LOW","MEDIUM","HIGH","CRITICAL"], fill_value=0)
display(risk_summary.to_frame("systems"))
display(analyzed[["health_score","cpu_pct","memory_pct","disk_pct","network_latency_ms","error_rate_pct"]].describe().T)

In [ ]:
risk_summary.plot(kind="bar", figsize=(8,4), title="System Risk Distribution")
plt.ylabel("Observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Regional and service analysis

In [ ]:
regional = analyzed.groupby("region").agg(
    systems=("system_id","count"),
    avg_health=("health_score","mean"),
    avg_latency=("network_latency_ms","mean"),
    avg_error_rate=("error_rate_pct","mean"),
    incidents=("incident_count","sum")
).sort_values("avg_health")
display(regional.round(2))

service = analyzed.groupby("service")["health_score"].agg(["count","mean","min"]).sort_values("mean")
display(service.round(2))

## 8. Executive summary

In [ ]:
critical = int((analyzed["risk_level"] == "CRITICAL").sum())
high = int((analyzed["risk_level"] == "HIGH").sum())
anomalies = int(analyzed["anomaly_flag"].sum())
avg_health = analyzed["health_score"].mean()

top_risk_metric = {
    "Disk utilization": analyzed["disk_pct"].mean(),
    "Network latency": analyzed["network_latency_ms"].mean(),
    "Error rate": analyzed["error_rate_pct"].mean(),
}.get(max(
    ["Disk utilization","Network latency","Error rate"],
    key=lambda x: {"Disk utilization": analyzed["disk_pct"].mean(),
                   "Network latency": analyzed["network_latency_ms"].mean()/10,
                   "Error rate": analyzed["error_rate_pct"].mean()*10}[x]
))

print("="*55)
print("ENTERPRISE SYSTEM HEALTH REPORT")
print("="*55)
print(f"Observations analyzed : {len(analyzed):,}")
print(f"Average health score  : {avg_health:.1f}/100")
print(f"High-risk systems     : {high:,}")
print(f"Critical systems      : {critical:,}")
print(f"Anomalous observations: {anomalies:,}")
print("="*55)
print("Interpretation:")
print("Prioritize investigation of systems classified HIGH or CRITICAL.")
print("Review anomaly observations for corroborating telemetry before remediation.")
print("Use the scoring thresholds as a portfolio/demo framework, not a production SLA.")


## 9. Key operational questions

1. Which systems require immediate investigation?
2. Are high error rates concentrated in a specific region or service?
3. Do latency anomalies coincide with application errors?
4. Which resource dimension contributes most to low health scores?
5. What additional telemetry would be needed before production deployment?

### Next steps

- Add time-series forecasting for capacity planning.
- Connect to Prometheus, Datadog, CloudWatch, or another telemetry source.
- Replace demonstration thresholds with organization-specific SLO/SLA thresholds.
- Add model-based anomaly detection and alert routing.
